In [ ]:
"""
PFD-Net V1 test-A — 二维 Poisson 方程求解（无监督 PINN，三阶段）
==================================================================
PDE:   -Δu(x,y) = f(x,y),   (x,y) ∈ Ω = [-1,1]²
边界:   u = g_b(x,y),        (x,y) ∈ ∂Ω   (Dirichlet)

精确解: F(x,y) = g(x) + g(y),  g(x) = exp(-x²) sin(μx²)
源项:   f(x,y) = -(g''(x) + g''(y))     ← 满足 -Δu = f
边界值: g_b = F(x,y)|∂Ω

网络结构（test-A，无自适应子域分解）:
  低频 APINN: 固定傅里叶嵌入（低频频率）+ tanh MLP
  高频 APINN: 固定傅里叶嵌入（高频频率）+ tanh MLP，全域覆盖
  PFDNet = 低频 APINN + 高频 APINN

三阶段训练（全程无监督，均使用 PINN 损失）:
  Phase 1 — 低频 APINN 预训练:
      仅训练 apinn_low，用 PDE残差 + BC 损失
  Phase 2 — 高频 APINN 预热（冻结低频）:
      冻结 apinn_low，仅训练 high_net
      损失 = PDE残差(整个pfdnet) + BC
  Phase 3 — 全参数联合训练:
      解冻所有参数，损失 = PDE残差 + BC

损失函数（各阶段统一）:
    L = λ_r · MSE[ -(u_xx + u_yy) - f ]  +  λ_b · MSE[ u|∂Ω - g_b ]

评估: 100×100 均匀测试网格，计算相对 L2 误差（与精确解对比，不用监督标签）
保存: 权重文件 + 训练曲线 (res_loss, bc_loss, total_loss, l2_error)
"""

import os
import time
import warnings
import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ── 设备 ──────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
torch.manual_seed(42)
np.random.seed(42)

CHECKPOINT_DIR = "checkpoints_test_A_pinn"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MU       = 30.0
LAMBDA_R = 1.0
LAMBDA_B = 10.0
pi       = torch.tensor(np.pi, dtype=torch.float64, device=device)


# ==============================================================================
# 精确解 & 源项
# ==============================================================================

def g_func(x: torch.Tensor) -> torch.Tensor:
    """g(x) = exp(-x²) sin(μx²)"""
    return torch.exp(-x ** 2) * torch.sin(MU * x ** 2)


def g_second(x: torch.Tensor) -> torch.Tensor:
    """
    g''(x) = e^{-x²} [(4x² - 4μ²x² - 2) sin(μx²) + (2μ - 8μx²) cos(μx²)]
    推导见 pinn_poisson_fixed.py 注释。
    """
    ex      = torch.exp(-x ** 2)
    s       = torch.sin(MU * x ** 2)
    c       = torch.cos(MU * x ** 2)
    coeff_s = 4 * x**2 - 4 * MU**2 * x**2 - 2
    coeff_c = 2 * MU - 8 * MU * x**2           # 修复后的正确系数
    return ex * (coeff_s * s + coeff_c * c)


def u_exact(xy: torch.Tensor) -> torch.Tensor:
    """精确解 F(x,y) = g(x) + g(y)；xy: (N,2) → (N,1)"""
    return g_func(xy[:, 0:1]) + g_func(xy[:, 1:2])


def f_source(xy: torch.Tensor) -> torch.Tensor:
    """源项 f，满足 -Δu = f：f = -(g''(x) + g''(y))"""
    return -(g_second(xy[:, 0:1]) + g_second(xy[:, 1:2]))


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    return u_exact(xy)


# ==============================================================================
# 网络结构
# ==============================================================================

class FixedFourierEmbed2D(nn.Module):
    """固定傅里叶嵌入（频率不参与训练），输出维度 = 4 * len(scales)"""
    def __init__(self, scales):
        super().__init__()
        self.register_buffer(
            "scales",
            torch.tensor(scales, dtype=torch.float64)
        )

    def get_scales(self):
        return self.scales.detach().cpu().numpy()

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        feats = []
        for s in self.scales:
            feats += [
                torch.sin(2 * pi * s * xy[:, 0:1]),
                torch.cos(2 * pi * s * xy[:, 0:1]),
                torch.sin(2 * pi * s * xy[:, 1:2]),
                torch.cos(2 * pi * s * xy[:, 1:2]),
            ]
        return torch.cat(feats, dim=-1)


class APINN2D(nn.Module):
    """统一 APINN 网络（低频/高频通用，固定傅里叶频率 + tanh MLP）"""
    def __init__(self, hidden_dim: int = 128, num_layers: int = 4,
                 scales=(1, 2, 4)):
        super().__init__()
        self.embed  = FixedFourierEmbed2D(scales)
        in_dim      = 2 + 4 * len(scales)
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, 1))
        for m in self.layers:
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        h = torch.cat([xy, self.embed(xy)], dim=-1)
        for layer in self.layers[:-1]:
            h = torch.tanh(layer(h))
        return self.layers[-1](h)


class PFDNet2D_TestA(nn.Module):
    """test-A：低频 APINN + 全域高频 APINN（无子域分解）"""
    def __init__(self, apinn_low: APINN2D, high_scales,
                 hidden_dim: int = 128, num_layers: int = 4):
        super().__init__()
        self.apinn_low = apinn_low
        self.high_net  = APINN2D(hidden_dim, num_layers, high_scales)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.apinn_low(xy) + self.high_net(xy)

    def print_scales(self, prefix: str = ""):
        print(f"{prefix}低频APINN 固定频率: "
              f"{np.round(self.apinn_low.embed.get_scales(), 2)}")
        print(f"{prefix}高频APINN(全域) 固定频率: "
              f"{np.round(self.high_net.embed.get_scales(), 2)}")


# ==============================================================================
# 自动微分：Δu = u_xx + u_yy
#
# 规则：laplacian 函数本身不碰 requires_grad。
#       调用前在循环内 xy_int.clone().requires_grad_(True) 创建干净入口。
# ==============================================================================

def laplacian(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    """
    xy: (N,2)，调用前已设 requires_grad=True
    返回: Δu = u_xx + u_yy，shape (N,1)
    """
    u = model(xy)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]                                    # (N,2)

    u_xx = torch.autograd.grad(
        grad_u[:, 0:1], xy,
        grad_outputs=torch.ones_like(grad_u[:, 0:1]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 0:1]                            # (N,1)

    u_yy = torch.autograd.grad(
        grad_u[:, 1:2], xy,
        grad_outputs=torch.ones_like(grad_u[:, 1:2]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 1:2]                            # (N,1)

    return u_xx + u_yy


# ==============================================================================
# 采样
# ==============================================================================

def sample_interior(n: int, seed: int = 1) -> torch.Tensor:
    """
    [-1,1]² 内部混合采样（均匀网格 + 随机），覆盖高频振荡区域。
    返回普通张量，不设 requires_grad（由训练循环 clone 后设置）。
    """
    rng  = np.random.RandomState(seed)
    side = int(np.sqrt(n // 2))
    gx   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    gy   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    GX, GY  = np.meshgrid(gx, gy)
    xy_grid = np.stack([GX.ravel(), GY.ravel()], axis=1)
    n_rand  = n - xy_grid.shape[0]
    xy_rand = rng.uniform(-1.0, 1.0, size=(n_rand, 2))
    pts     = np.concatenate([xy_grid, xy_rand], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


def sample_boundary(n_per_edge: int, seed: int = 2) -> torch.Tensor:
    """四条边各 n_per_edge 个点，返回 (4*n_per_edge, 2)"""
    rng    = np.random.RandomState(seed)
    t      = rng.uniform(-1.0, 1.0, n_per_edge)
    bottom = np.stack([t,  -np.ones(n_per_edge)], axis=1)
    top    = np.stack([t,   np.ones(n_per_edge)], axis=1)
    left   = np.stack([-np.ones(n_per_edge), t],  axis=1)
    right  = np.stack([ np.ones(n_per_edge), t],  axis=1)
    pts    = np.concatenate([bottom, top, left, right], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


# ==============================================================================
# PINN 损失计算（各阶段复用）
# ==============================================================================

def pinn_loss(model: nn.Module,
              xy_int: torch.Tensor,
              f_int:  torch.Tensor,
              xy_bc:  torch.Tensor,
              u_bc:   torch.Tensor,
              lambda_r: float,
              lambda_b: float):
    """
    计算 PDE残差损失 + 边界损失，返回 (total, loss_r, loss_b)。

    xy_int: 普通张量（函数内部 clone + requires_grad_(True)）
    f_int:  源项，已 detach
    xy_bc:  边界点，普通张量
    u_bc:   边界真值，已 detach
    """
    # PDE 残差
    xy_r  = xy_int.clone().requires_grad_(True)
    lap_u = laplacian(model, xy_r)              # Δu
    loss_r = torch.mean((-lap_u - f_int) ** 2)  # -Δu - f = 0

    # 边界条件
    loss_b = nn.functional.mse_loss(model(xy_bc), u_bc)

    total = lambda_r * loss_r + lambda_b * loss_b
    return total, loss_r, loss_b


# ==============================================================================
# 三阶段训练（全程无监督 PINN）
# ==============================================================================

def train_test_A(
    # 网络超参
    hidden_dim     : int   = 128,
    num_layers     : int   = 4,
    low_scales              = (1, 2, 4),
    high_scales             = (8, 16, 32),
    # 配点
    n_interior     : int   = 10000,
    n_per_edge     : int   = 250,
    # 训练轮次（三阶段之和 = 15000）
    pretrain_epochs: int   = 8000,
    warmup_epochs  : int   = 2000,
    joint_epochs   : int   = 5000,
    # 学习率
    lr_pretrain    : float = 5e-3,
    lr_warmup      : float = 1e-3,
    lr_joint       : float = 5e-4,
    # 损失权重
    lambda_r       : float = LAMBDA_R,
    lambda_b       : float = LAMBDA_B,
    # 日志
    log_every      : int   = 1,
):
    total_epochs = pretrain_epochs + warmup_epochs + joint_epochs
    assert total_epochs == 15000, \
        f"三阶段之和应为 15000，当前为 {total_epochs}"

    # ── 固定配点 ──────────────────────────────────────────────────────────────
    xy_int = sample_interior(n_interior, seed=1)     # 普通张量
    f_int  = f_source(xy_int).detach()               # 源项，固定

    xy_bc  = sample_boundary(n_per_edge, seed=2)
    u_bc   = g_boundary(xy_bc).detach()

    print(f"内部配点: {xy_int.shape[0]},  边界配点: {xy_bc.shape[0]}")

    # ── 测试集（100×100 均匀网格，与精确解对比）────────────────────────────
    nx      = 100
    xv, yv  = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, nx))
    xy_test = torch.tensor(
        np.stack([xv.ravel(), yv.ravel()], axis=1),
        dtype=torch.float64, device=device
    )
    u_test         = u_exact(xy_test).detach()
    u_test_sq_mean = torch.mean(u_test ** 2).item()

    # ── 评估辅助 ──────────────────────────────────────────────────────────────
    def eval_l2(model: nn.Module) -> float:
        with torch.no_grad():
            pred = model(xy_test)
            return torch.sqrt(
                torch.mean((pred - u_test) ** 2) / u_test_sq_mean
            ).item()

    # ── 全局记录 ──────────────────────────────────────────────────────────────
    epochs_record  = []
    res_losses     = []
    bc_losses      = []
    total_losses   = []
    l2_errors      = []
    epoch_offset   = 0

    def record_and_print(ep_local, total_l, loss_r, loss_b, model, tag):
        ep_global = ep_local + epoch_offset
        l2        = eval_l2(model)
        epochs_record.append(ep_global)
        res_losses.append(loss_r.item())
        bc_losses.append(loss_b.item())
        total_losses.append(total_l.item())
        l2_errors.append(l2)
        if ep_local % max(1, log_every * 5) == 0 or ep_local == 1:
            print(
                f"  {tag} {ep_local:6d} | "
                f"res: {loss_r.item():.3e} | "
                f"bc: {loss_b.item():.3e} | "
                f"total: {total_l.item():.3e} | "
                f"L2: {l2:.6f} | {time.time()-t0:.1f}s"
            )

    t0 = time.time()

    # ==========================================================================
    # Phase 1：低频 APINN 预训练（PINN 无监督）
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 1: 低频APINN 预训练（PINN）| {pretrain_epochs} epochs")
    print(f"  低频频率: {list(low_scales)}  lr={lr_pretrain:.2e}")
    print(f"  λ_r={lambda_r}, λ_b={lambda_b}")
    print(f"{'='*60}")

    apinn_low    = APINN2D(hidden_dim, num_layers, low_scales).double().to(device)
    total_p1     = sum(p.numel() for p in apinn_low.parameters())
    print(f"低频APINN 参数量: {total_p1:,}")

    opt1 = torch.optim.Adam(apinn_low.parameters(), lr=lr_pretrain)
    # sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt1, pretrain_epochs, eta_min=1e-5)

    for ep in range(1, pretrain_epochs + 1):
        apinn_low.train()
        opt1.zero_grad()
        loss, loss_r, loss_b = pinn_loss(
            apinn_low, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(apinn_low.parameters(), 1.0)
        opt1.step()
        # sch1.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, apinn_low, "pretrain")

    torch.save(apinn_low.state_dict(),
               os.path.join(CHECKPOINT_DIR, "phase1_apinn_low.pt"))
    print(f"\n  Phase 1 结束 | L2={eval_l2(apinn_low):.6f} | "
          f"低频频率: {np.round(apinn_low.embed.get_scales(), 3)}")
    epoch_offset += pretrain_epochs

    # ==========================================================================
    # Phase 2：高频 APINN 预热（冻结低频，PINN 无监督）
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 2: 高频APINN 预热（冻结低频）| {warmup_epochs} epochs")
    print(f"  高频频率: {list(high_scales)}  lr={lr_warmup:.2e}")
    print(f"{'='*60}")

    # 构建完整 PFDNet（低频 + 高频），冻结低频参数
    pfdnet = PFDNet2D_TestA(
        apinn_low, high_scales, hidden_dim, num_layers
    ).double().to(device)
    total_p2 = sum(p.numel() for p in pfdnet.parameters())
    print(f"PFDNet 总参数量: {total_p2:,}")

    for p in pfdnet.apinn_low.parameters():
        p.requires_grad = False

    high_params = list(pfdnet.high_net.parameters())

    opt2 = torch.optim.Adam(high_params, lr=lr_warmup)
    # sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt2, warmup_epochs, eta_min=1e-5)

    for ep in range(1, warmup_epochs + 1):
        pfdnet.train()
        opt2.zero_grad()
        # PDE残差对整个 pfdnet 求（低频 + 高频输出之和）
        loss, loss_r, loss_b = pinn_loss(
            pfdnet, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(high_params, 1.0)
        opt2.step()
        # sch2.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, pfdnet, "warmup ")

    torch.save(pfdnet.state_dict(),
               os.path.join(CHECKPOINT_DIR, "phase2_pfdnet.pt"))
    print(f"\n  Phase 2 结束 | L2={eval_l2(pfdnet):.6f}")
    epoch_offset += warmup_epochs

    # ==========================================================================
    # Phase 3：全参数联合训练（PINN 无监督）
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 3: 全参数联合训练（PINN）| {joint_epochs} epochs")
    print(f"  lr={lr_joint:.2e}")
    print(f"{'='*60}")

    # 解冻低频参数
    for p in pfdnet.apinn_low.parameters():
        p.requires_grad = True

    opt3 = torch.optim.Adam([
        {'params': pfdnet.apinn_low.parameters(), 'lr': lr_joint},
        {'params': pfdnet.high_net.parameters(),  'lr': lr_joint},
    ])
    # sch3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt3, joint_epochs, eta_min=1e-6)

    for ep in range(1, joint_epochs + 1):
        pfdnet.train()
        opt3.zero_grad()
        loss, loss_r, loss_b = pinn_loss(
            pfdnet, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(list(pfdnet.parameters()), 1.0)
        opt3.step()
        # sch3.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, pfdnet, "joint  ")

    # ── 最终指标 ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        fp   = pfdnet(xy_test)
        fl2  = torch.sqrt(
            torch.mean((fp - u_test) ** 2) / u_test_sq_mean).item()
        fmae = torch.mean(torch.abs(fp - u_test)).item()
    total_time = time.time() - t0

    print(f"\n{'='*60}")
    print(f"  训练完成 | L2: {fl2:.6f} | MAE: {fmae:.6f} | 时间: {total_time:.1f}s")
    pfdnet.print_scales("  ")
    print(f"{'='*60}")

    # ── 保存 ──────────────────────────────────────────────────────────────────
    torch.save(pfdnet.state_dict(),
               os.path.join(CHECKPOINT_DIR, "testA_pinn_weights.pt"))
    np.savez(
        os.path.join(CHECKPOINT_DIR, "testA_pinn_curves.npz"),
        epochs     = np.array(epochs_record),
        res_loss   = np.array(res_losses),
        bc_loss    = np.array(bc_losses),
        total_loss = np.array(total_losses),
        l2_error   = np.array(l2_errors),
    )
    print(f"权重与曲线已保存至 {CHECKPOINT_DIR}/")

    return dict(
        model=pfdnet,
        xy_test=xy_test, u_test=u_test,
        epochs=np.array(epochs_record),
        res_loss=np.array(res_losses),
        bc_loss=np.array(bc_losses),
        total_loss=np.array(total_losses),
        l2_error=np.array(l2_errors),
        fl2=fl2, fmae=fmae,
        hidden_dim=hidden_dim, num_layers=num_layers,
        total_params=total_p2,
        n_interior=n_interior, n_bc=xy_bc.shape[0],
        low_scales=low_scales, high_scales=high_scales,
    )


# ==============================================================================
if __name__ == "__main__":
    train_test_A(
        hidden_dim      = 128,
        num_layers      = 4,
        low_scales      = (1, 2, 4),
        high_scales     = (8, 16, 32),
        n_interior      = 10000,
        n_per_edge      = 250,
        pretrain_epochs = 8000,
        warmup_epochs   = 2000,
        joint_epochs    = 5000,
        lr_pretrain     = 5e-3,
        lr_warmup       = 1e-3,
        lr_joint        = 5e-4,
        lambda_r        = 1.0,
        lambda_b        = 10.0,
        log_every       = 1,
    )